# ITMS M8 — fire and smoke crops from D-Fire's test images (for a laptop check)

The fire detector (version 2) passed its bars, but it scores Bengaluru traffic frames almost at zero
while it alarms on 28 % of D-Fire's own no-fire images at the same threshold. That could mean it learned
"a Bengaluru CCTV picture is never a fire" instead of "orange autos are not fire". To check, the laptop
pastes real flames and smoke into Bengaluru frames and sees whether the detector still finds them.

This notebook only cuts the flames and smoke out of D-Fire's **test** images (which no model trained on)
and zips them. It needs **no GPU** and takes about 5-10 minutes.

It takes the test images from the fire notebook's prepared dataset in Drive (`itms_fire_v2/fire_dataset.tar`)
if that is still there, and otherwise downloads the same Kaggle copy of D-Fire the fire notebook used
(the Kaggle key in Colab's Secrets, as before; allow this notebook to use it if Colab asks).

**How to run:** Runtime → Run all → allow Google Drive. At the end `fire_crops.zip` downloads (it is also
in Drive, `itms_fire_v2`). Unzip it anywhere on the laptop and tell Claude where.

In [ ]:
# ---- Settings ----
DRIVE_DIR = "/content/drive/MyDrive/itms_fire_v2"   # where the fire notebook (version 2) kept its dataset
PER_CLASS = 400        # crops of each class (fire, smoke), chosen at random with a fixed seed
MIN_SIDE = 24          # a box smaller than this (px, either side) is too small to paste
MARGIN = 0.10          # context kept around each box, as a share of its size
SEED = 26
KAGGLE_DATASET = "sayedgamal99/smoke-fire-detection-yolo"   # the copy the fire notebook used
KAGGLE_NAMES = {0: "smoke", 1: "fire"}   # its label ids, as the fire notebook recorded (and colour-checked)
PREPARED_NAMES = {0: "fire", 1: "smoke"}   # the ids in the fire notebook's own prepared dataset
N_TEST = 4306          # test images the fire notebook scored; anything else is not the same test split

In [ ]:
# ---- The test split: from the prepared dataset in Drive, or the same Kaggle download ----
import os, re, json, random, shutil, subprocess, zipfile, glob
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
WORK = "/content/crops_work"
os.makedirs(WORK, exist_ok=True)
TAR = f"{DRIVE_DIR}/fire_dataset.tar"
if os.path.exists(TAR):
    if not os.path.isdir(f"{WORK}/ds/images/test"):
        subprocess.run(["tar", "-xf", TAR, "-C", WORK, "ds/images/test", "ds/labels/test"], check=True)
    img_dir, lab_dir, ids = f"{WORK}/ds/images/test", f"{WORK}/ds/labels/test", PREPARED_NAMES
    SOURCE = "the fire notebook's prepared dataset in Drive"
else:
    print(f"{TAR} is not in Drive; downloading the Kaggle copy of D-Fire instead ...")
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:  # noqa: BLE001
                pass
    except Exception:  # noqa: BLE001
        pass
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub
    root = kagglehub.dataset_download(KAGGLE_DATASET)
    tests = [d for d in glob.glob(f"{root}/**/test/images", recursive=True) if os.path.isdir(d)]
    assert len(tests) == 1, f"expected one test/images folder in the download, found {tests}"
    img_dir = tests[0]
    lab_dir = os.path.join(os.path.dirname(img_dir), "labels")
    ids = KAGGLE_NAMES
    for y in glob.glob(f"{root}/**/*.yaml", recursive=True):   # the download's own names must agree
        m = re.search(r"names\s*:\s*\[([^\]]*)\]", open(y, errors="replace").read())
        if m:
            declared = [v.strip().strip("'\"").lower() for v in m.group(1).split(",") if v.strip()]
            assert {i: n for i, n in enumerate(declared)} == KAGGLE_NAMES, (
                f"{y} declares {declared}; the fire notebook recorded {KAGGLE_NAMES}. Stop and tell Claude.")
    SOURCE = f"Kaggle {KAGGLE_DATASET}"
imgs = sorted(p for p in glob.glob(f"{img_dir}/*") if p.lower().endswith((".jpg", ".jpeg", ".png")))
print(len(imgs), "test images from", SOURCE)
assert len(imgs) == N_TEST, (f"{len(imgs)} test images, the fire notebook scored {N_TEST}: not the same test "
                             "split. Stop and tell Claude.")

In [ ]:
# ---- Cut out every usable box, then keep PER_CLASS of each class ----
import cv2
boxes = {"fire": [], "smoke": []}
for p in imgs:
    stem = os.path.splitext(os.path.basename(p))[0]
    lab = f"{lab_dir}/{stem}.txt"
    if not os.path.exists(lab):
        continue
    for i, line in enumerate(open(lab)):
        parts = line.split()
        if len(parts) != 5:
            continue
        c, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
        if c in ids:
            boxes[ids[c]].append((p, stem, i, cx, cy, w, h))
rng = random.Random(SEED)
OUT = f"{WORK}/fire_crops"
os.makedirs(OUT, exist_ok=True)
index = []
for c, lst in boxes.items():
    rng.shuffle(lst)
    kept = 0
    for p, stem, i, cx, cy, w, h in lst:
        if kept >= PER_CLASS:
            break
        img = cv2.imread(p)
        if img is None:
            continue
        H, W = img.shape[:2]
        bw, bh = w * W, h * H
        if min(bw, bh) < MIN_SIDE:
            continue
        x1 = max(0, int(cx * W - bw / 2 - MARGIN * bw))
        y1 = max(0, int(cy * H - bh / 2 - MARGIN * bh))
        x2 = min(W, int(cx * W + bw / 2 + MARGIN * bw))
        y2 = min(H, int(cy * H + bh / 2 + MARGIN * bh))
        name = f"{c}_{stem}_{i}.png"
        cv2.imwrite(f"{OUT}/{name}", img[y1:y2, x1:x2])
        index.append({"file": name, "class": c, "source": stem, "box_px": [round(bw), round(bh)]})
        kept += 1
    print(c, ": kept", kept, "of", len(lst), "boxes")
    assert kept >= min(100, PER_CLASS), f"only {kept} usable {c} boxes: stop and tell Claude"
json.dump({"seed": SEED, "per_class": PER_CLASS, "min_side": MIN_SIDE, "margin": MARGIN, "source": SOURCE,
           "split": "D-Fire test (never trained on)", "crops": index}, open(f"{OUT}/crops.json", "w"), indent=1)

In [ ]:
# ---- Zip and download ----
zp = f"{DRIVE_DIR}/fire_crops.zip"
with zipfile.ZipFile(zp + ".part", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(f"{OUT}/{fn}", f"fire_crops/{fn}")
os.replace(zp + ".part", zp)
print("saved", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
try:
    from google.colab import files as _files
    _files.download(zp)
except Exception:  # noqa: BLE001
    print("Download it from Drive: folder itms_fire_v2")